# BOLD5000 fMRI Data Analysis
After bad results with GAN, MLP, we analyze our data. 

In [29]:
import os
import zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [30]:
# Constants
SUBJECT = "CSI4" # Configurable to switch subjects
HDD_DIR = "/media/hdd/BOLD5000"
PROCESSED_DIR = "./processed_data"

# Ensure processed directory exists for the subject
os.makedirs(os.path.join(PROCESSED_DIR, SUBJECT), exist_ok=True)

# List of target visual ROIs we want to extract
# From the GLMsingle ROI betas, we saw regions like LHEarlyVis, RHEarlyVis, LHLOC, RHLOC, LHOPA, RHOPA, LHPPA, RHPPA, LHRSC, RHRSC
TARGET_ROIS = [
    "LHEarlyVis", "RHEarlyVis",
    "LHLOC", "RHLOC",
    "LHOPA", "RHOPA",
    "LHPPA", "RHPPA",
    "LHRSC", "RHRSC"
]

# We are using TYPE D which id best estimate per paper
BETA_TYPE = "TYPED-FITHRF-GLMDENOISE-RR"

## Preprocess Data ##

In [ ]:
# Extract relevant ROIs directly from the ZIP
import zipfile
import re
import json

# --- Label mapping helpers (mirrors preprocess.py) ---
def _load_imagenet_synset_to_label(mappings_dir: str) -> dict:
    inet_path = os.path.join(mappings_dir, "imagenet_class_index.json")
    if not os.path.exists(inet_path):
        raise FileNotFoundError(
            f"Missing {inet_path}. Run preprocess.py once (it downloads it), "
            "or place imagenet_class_index.json in label_mappings/."
        )
    with open(inet_path, "r") as f:
        raw = json.load(f)
    # raw: idx -> [synset, label]
    return {v[0]: v[1] for v in raw.values()}


def _load_coco_img_to_categories(mappings_dir: str) -> dict:
    coco_path = os.path.join(mappings_dir, "instances_train2014.json")
    if not os.path.exists(coco_path):
        raise FileNotFoundError(
            f"Missing {coco_path}. Run preprocess.py once (it downloads it), "
            "or place instances_train2014.json in label_mappings/."
        )
    with open(coco_path, "r") as f:
        coco_data = json.load(f)
    cat_id_to_name = {cat["id"]: cat["name"] for cat in coco_data["categories"]}
    img_to_cats = {}
    for ann in coco_data["annotations"]:
        img_id = ann["image_id"]
        cat_name = cat_id_to_name.get(ann["category_id"], "unknown")
        if img_id not in img_to_cats:
            img_to_cats[img_id] = []
        if cat_name not in img_to_cats[img_id]:
            img_to_cats[img_id].append(cat_name)
    return img_to_cats


def _resolve_labels_for_filename(filename: str, imagenet_synset_to_label: dict, coco_img_to_categories: dict):
    """Return (dataset_source, labels:list[str]) for a stimulus filename."""
    basename = os.path.basename(filename)

    # ImageNet: starts with n followed by digits, then underscore
    if re.match(r'^n\d+_', basename):
        synset = basename.split('_')[0]
        label = imagenet_synset_to_label.get(synset, synset)
        return "ImageNet", [label]

    # COCO: filename contains 'COCO'
    if "COCO" in basename:
        m = re.search(r'(\d{6,12})', basename)
        img_id = int(m.group(1)) if m else -1
        cats = coco_img_to_categories.get(img_id, [])
        return "COCO", cats if cats else [f"COCO image {img_id}"]

    # Scene / SUN / other
    scene = os.path.splitext(basename)[0]
    scene = re.sub(r'\d+', '', scene)
    scene = re.sub(r'([a-z])([A-Z])', r'\1 \2', scene)
    return "Scene", [scene.strip().title() or basename]


def build_trial_labels(img_names: list[str]):
    """Build arrays: dataset_sources (N,), labels (N,) (object list[str])."""
    mappings_dir = os.path.join("..", "label_mappings")
    imagenet_synset_to_label = _load_imagenet_synset_to_label(mappings_dir)
    coco_img_to_categories = _load_coco_img_to_categories(mappings_dir)

    dataset_sources = []
    labels = []
    for name in img_names:
        src, lbs = _resolve_labels_for_filename(name, imagenet_synset_to_label, coco_img_to_categories)
        dataset_sources.append(src)
        labels.append(lbs)
    return np.array(dataset_sources), np.array(labels, dtype=object)


def parse_and_save_rois(subject):
    print(f"Extracting and merging ROI data for {subject}...")
    zip_path = os.path.join(HDD_DIR, "BOLD5000_GLMsingle_ROI_betas.zip")

    subject_rois = {}

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Get all relevant .npy files
        files = [f for f in z.namelist() if f.endswith('.npy') and f"/{subject}_" in f and BETA_TYPE in f]

        for roi in TARGET_ROIS:
            roi_file = next((f for f in files if roi in f), None)
            if roi_file:
                with z.open(roi_file) as f:
                    roi_data = np.load(f)
                    subject_rois[roi] = roi_data
                    print(f"Loaded {roi} | Shape: {roi_data.shape}")
            else:
                print(f"Warning: {roi} not found for {subject}")

    # Concatenate all ROI matrices horizontally (Trial x Voxel)
    concat_list = []
    roi_indices_map = {}
    current_idx = 0

    for roi in TARGET_ROIS:
        if roi in subject_rois:
            mat = subject_rois[roi]
            num_voxels = mat.shape[1]
            concat_list.append(mat)
            roi_indices_map[roi] = (current_idx, current_idx + num_voxels)
            current_idx += num_voxels

    all_visual_voxels = np.concatenate(concat_list, axis=1)
    print(f"\nFinal Concatenated Shape ({subject}): {all_visual_voxels.shape}")

    # Load stimulus order for this subject
    imgnames_path = os.path.join(HDD_DIR, f"{subject}_imgnames.txt")
    with open(imgnames_path, 'r') as f:
        img_names = [line.strip() for line in f.readlines()]
    print(f"Loaded {len(img_names)} image names for {subject}.")

    # NEW: build labels + dataset sources and save them alongside the ROI betas
    print("Building per-trial labels (ImageNet/COCO/Scene)...")
    dataset_sources, labels = build_trial_labels(img_names)
    print("Label mapping complete.")

    # Save into ./processed_data/<participant>/ (e.g., ./processed_data/CSI1/CSI1_visual_rois.npz)
    out_dir = os.path.join(PROCESSED_DIR, subject)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{subject}_visual_rois.npz")
    np.savez_compressed(
        out_path,
        betas=all_visual_voxels,
        imgnames=np.array(img_names),
        roi_mapping=roi_indices_map,
        dataset_sources=dataset_sources,
        labels=labels,
    )
    print(f"Saved optimized data to: {out_path}\n")
    return all_visual_voxels, img_names, roi_indices_map

# Run extraction for our selected target subject
betas, imgnames, roi_map = parse_and_save_rois(SUBJECT)

Extracting and merging ROI data for CSI4...
Loaded LHEarlyVis | Shape: (3108, 408)
Loaded RHEarlyVis | Shape: (3108, 356)
Loaded LHLOC | Shape: (3108, 455)
Loaded RHLOC | Shape: (3108, 417)
Loaded LHOPA | Shape: (3108, 279)
Loaded RHOPA | Shape: (3108, 335)
Loaded LHPPA | Shape: (3108, 157)
Loaded RHPPA | Shape: (3108, 187)
Loaded LHRSC | Shape: (3108, 51)
Loaded RHRSC | Shape: (3108, 142)

Final Concatenated Shape (CSI4): (3108, 2787)
Loaded 3108 image names for CSI4.
Building per-trial labels (ImageNet/COCO/Scene)...
Label mapping complete.
Label mapping complete.
Saved optimized data to: ./processed_data/CSI4/CSI4_visual_rois.npz

Saved optimized data to: ./processed_data/CSI4/CSI4_visual_rois.npz



In [32]:
# --- Sanity-check saved processed ROI files for all subjects (CSI1–CSI4) ---
import numpy as np
from collections import Counter

def _load_roi_npz(subject: str, processed_root: str = "./processed_data"):
    path = os.path.join(processed_root, subject, f"{subject}_visual_rois.npz")
    if not os.path.exists(path):
        return path, None
    return path, np.load(path, allow_pickle=True)

def _roi_list(roi_mapping):
    if roi_mapping is None:
        return []
    if isinstance(roi_mapping, np.ndarray):  # sometimes comes as 0-d array containing a dict
        try:
            roi_mapping = roi_mapping.item()
        except Exception:
            pass
    if isinstance(roi_mapping, dict):
        return sorted(list(roi_mapping.keys()))
    return []

subjects = ["CSI1", "CSI2", "CSI3", "CSI4"]
loaded = {}
for s in subjects:
    path, npz = _load_roi_npz(s, processed_root=PROCESSED_DIR)
    loaded[s] = (path, npz)

print("=" * 90)
print("Processed ROI files (expected: one per subject in ./processed_data/<SUBJECT>/)")
print("=" * 90)
for s in subjects:
    path, npz = loaded[s]
    print(f"\n{s}: {path}")
    if npz is None:
        print("  [MISSING] not found")
        continue
    keys = list(npz.keys())
    betas = npz["betas"] if "betas" in npz else None
    imgnames = npz["imgnames"] if "imgnames" in npz else None
    labels = npz["labels"] if "labels" in npz else None
    sources = npz["dataset_sources"] if "dataset_sources" in npz else None
    roi_mapping = npz["roi_mapping"] if "roi_mapping" in npz else None

    print(f"  keys: {keys}")
    if betas is not None:
        print(f"  betas   : {betas.shape}")
    if imgnames is not None:
        print(f"  imgnames: {imgnames.shape}")
    if labels is not None:
        print(f"  labels  : {labels.shape}  dtype={labels.dtype}")
    if sources is not None:
        try:
            c = Counter([str(x) for x in sources.tolist()])
            print("  sources : " + ", ".join([f"{k}={v}" for k, v in sorted(c.items())]))
        except Exception:
            print(f"  sources : {sources.shape}")

    rois = _roi_list(roi_mapping)
    print(f"  #ROIs   : {len(rois)}")
    if rois:
        print(f"  ROIs    : {rois}")

# Compare ROI presence across subjects (helps explain voxel-dim differences)
roi_sets = {s: set(_roi_list(loaded[s][1]["roi_mapping"]) if loaded[s][1] is not None and "roi_mapping" in loaded[s][1] else []) for s in subjects}
all_rois = sorted(set().union(*roi_sets.values()))
print("\n" + "=" * 90)
print("ROI presence table (1=present, 0=missing)")
print("=" * 90)
header = "ROI".ljust(20) + " " + " ".join([s.rjust(5) for s in subjects])
print(header)
print("-" * len(header))
for roi in all_rois:
    row = roi.ljust(20) + " " + " ".join([("1" if roi in roi_sets[s] else "0").rjust(5) for s in subjects])
    print(row)

Processed ROI files (expected: one per subject in ./processed_data/<SUBJECT>/)

CSI1: ./processed_data/CSI1/CSI1_visual_rois.npz
  keys: ['betas', 'imgnames', 'roi_mapping', 'dataset_sources', 'labels']
  betas   : (5254, 1685)
  imgnames: (5254,)
  labels  : (5254,)  dtype=object
  sources : COCO=2135, ImageNet=2051, Scene=1068
  #ROIs   : 10
  ROIs    : ['LHEarlyVis', 'LHLOC', 'LHOPA', 'LHPPA', 'LHRSC', 'RHEarlyVis', 'RHLOC', 'RHOPA', 'RHPPA', 'RHRSC']

CSI2: ./processed_data/CSI2/CSI2_visual_rois.npz
  keys: ['betas', 'imgnames', 'roi_mapping', 'dataset_sources', 'labels']
  betas   : (5254, 1104)
  imgnames: (5254,)
  labels  : (5254,)  dtype=object
  sources : COCO=2135, ImageNet=2051, Scene=1068
  #ROIs   : 7
  ROIs    : ['LHEarlyVis', 'LHOPA', 'LHPPA', 'LHRSC', 'RHEarlyVis', 'RHOPA', 'RHPPA']

CSI3: ./processed_data/CSI3/CSI3_visual_rois.npz
  keys: ['betas', 'imgnames', 'roi_mapping', 'dataset_sources', 'labels']
  betas   : (5254, 1466)
  imgnames: (5254,)
  labels  : (5254,) 

In [40]:
# Quickly check the RAW fMRI beta files in /media/hdd/BOLD5000 (one per subject/session)
# and see whether the *raw* data shape differs across participants.
# This helps decide if differences are inherent to the raw files vs our ROI preprocessing.

import os
import nibabel as nib
from collections import defaultdict

def raw_beta_path(subject: str, session: int) -> str:
    return os.path.join(
        HDD_DIR,
        f"{subject}_GLMbetas-TYPED-FITHRF-GLMDENOISE-RR_ses-{session:02d}.nii.gz",
    )

subjects = ["CSI1", "CSI2", "CSI3", "CSI4"]
subject_sessions = {
    "CSI1": list(range(1, 16)),
    "CSI2": list(range(1, 16)),
    "CSI3": list(range(1, 16)),
    "CSI4": list(range(1, 10)),  # CSI4 often has fewer sessions available
}

print("=" * 100)
print("RAW GLM beta NIfTI shapes per subject/session")
print("(format: (X, Y, Z, N_trials))")
print("=" * 100)

summary = {}
missing = []
for subj in subjects:
    seshs = subject_sessions.get(subj, [])
    shapes = []
    trial_counts = []
    xyz_shapes = set()
    print(f"\n{subj}:")
    for sesh in seshs:
        p = raw_beta_path(subj, sesh)
        if not os.path.exists(p):
            print(f"  ses-{sesh:02d}: [MISSING] {p}")
            missing.append((subj, sesh, p))
            continue
        try:
            img = nib.load(p)
            shp = tuple(img.shape)
        except Exception as e:
            print(f"  ses-{sesh:02d}: [ERROR] {e}")
            continue

        # Expect 4D beta volumes: (X, Y, Z, trials)
        if len(shp) != 4:
            print(f"  ses-{sesh:02d}: shape={shp}  [WARN] expected 4D")
            continue
        x, y, z, n_trials = shp
        print(f"  ses-{sesh:02d}: shape={shp}")
        shapes.append(shp)
        trial_counts.append(n_trials)
        xyz_shapes.add((x, y, z))

    # Per-subject summary
    if shapes:
        uniq_xyz = sorted(list(xyz_shapes))
        summary[subj] = {
            "n_sessions_found": len(shapes),
            "xyz_shapes": uniq_xyz,
            "trial_counts": trial_counts,
            "total_trials": int(sum(trial_counts)),
        }
        print("  --")
        print(f"  sessions found : {summary[subj]['n_sessions_found']} / {len(seshs)}")
        print(f"  unique XYZ grids: {uniq_xyz}")
        print(f"  total trials   : {summary[subj]['total_trials']}")
        if len(set(trial_counts)) > 1:
            # this is normal, but useful to see
            print(f"  trial count range: {min(trial_counts)}–{max(trial_counts)}")
    else:
        summary[subj] = {
            "n_sessions_found": 0,
            "xyz_shapes": [],
            "trial_counts": [],
            "total_trials": 0,
        }

print("\n" + "=" * 100)
print("COMPARISON SUMMARY")
print("=" * 100)
for subj in subjects:
    s = summary[subj]
    print(f"{subj}: sessions_found={s['n_sessions_found']} total_trials={s['total_trials']} xyz_grids={s['xyz_shapes']}")

if missing:
    print("\n" + "=" * 100)
    print(f"Missing files ({len(missing)}):")
    print("=" * 100)
    # print only a few examples to keep output readable
    for (subj, sesh, p) in missing[:10]:
        print(f"  {subj} ses-{sesh:02d}: {p}")
    if len(missing) > 10:
        print(f"  ... and {len(missing) - 10} more")

RAW GLM beta NIfTI shapes per subject/session
(format: (X, Y, Z, N_trials))

CSI1:
  ses-01: shape=(71, 89, 72, 370)
  ses-02: shape=(71, 89, 72, 370)
  ses-03: shape=(71, 89, 72, 370)
  ses-04: shape=(71, 89, 72, 333)
  ses-05: shape=(71, 89, 72, 370)
  ses-06: shape=(71, 89, 72, 333)
  ses-07: shape=(71, 89, 72, 370)
  ses-08: shape=(71, 89, 72, 333)
  ses-09: shape=(71, 89, 72, 333)
  ses-10: shape=(71, 89, 72, 370)
  ses-11: shape=(71, 89, 72, 333)
  ses-12: shape=(71, 89, 72, 333)
  ses-13: shape=(71, 89, 72, 333)
  ses-14: shape=(71, 89, 72, 333)
  ses-15: shape=(71, 89, 72, 370)
  --
  sessions found : 15 / 15
  unique XYZ grids: [(71, 89, 72)]
  total trials   : 5254
  trial count range: 333–370

CSI2:
  ses-01: shape=(72, 92, 70, 370)
  ses-02: shape=(72, 92, 70, 333)
  ses-03: shape=(72, 92, 70, 370)
  ses-04: shape=(72, 92, 70, 370)
  ses-05: shape=(72, 92, 70, 370)
  ses-06: shape=(72, 92, 70, 333)
  ses-07: shape=(72, 92, 70, 333)
  ses-08: shape=(72, 92, 70, 333)
  ses-09

We now see that the RAW data varies per subject
This leaves us with two main paths
1) Use the Schaefer regions which we compress into equal represenations for everyone
    * Benefits: We can compare everyone at once, use one autoencoder for all and makes our results more generalizeable
    * Downsides: This compression "loses" data and generalizes things across participants that might indeed not be generalizeable
    
2) Use per person autoencoder with either raw or personalized ROI data
    * Benefits: we guarantee to not "lose data" or generalize
    * We will have four of everything, stemming from individually different datasets. 

CONCLUSION:

4 of everything is bound to be a mess. We'll have little data per participant, and everything would ahve to be duplicated 4x. This is unnecceptable

We will use Schaefer Regions, the max, so 1000 + 14 subcortico, and use this for autonencoders. 
We will be able to have one general strucuture for everything : model + analysis, and we can still do per person analysis, but using the same structure